# Week 8 Homework: Hierarchical Clustering

## Purpose of Homework

This homework will give you practice applying **hierarchical clustering** to explore structure in a real-world dataset of recipes from around the world. Unlike supervised methods, clustering is **unsupervised**: the goal is to discover natural groupings in the data.

You will build and interpret a **dendrogram**, choose the number of clusters, profile each cluster using summary statistics and visualizations, and reflect on what the clusters reveal about global cuisine.

## Logistics

Due date: The homework is due **11:59pm on Thursday, March 26, 2026**.

You will submit your homework on [MarkUs](https://markus.teach.cs.toronto.edu/markus/).

1. Download this file (`STA272_hw8_student.ipynb`) from JupyterHub. 
2. Submit this file to MarkUs

All homeworks will take place in a Jupyter notebook (like this one). When you are done, you will download this notebook and submit it to MarkUs.

## About the Data

The **cuisines_clean** dataset contains recipes drawn from distinct world cuisines. Each recipe includes nutritional information, preparation and cooking times, user ratings, and country/cuisine of origin.

**Research Question:** Can we identify natural groups of recipes based on their nutritional profiles and preparation characteristics? Do these groups correspond to recognizable culinary traditions or dish types?

### Variables

| Variable | Description | Type |
|----------|-------------|------|
| `name` | Recipe name | Text |
| `country` | Cuisine origin | Categorical |
| `calories` | Total calories per serving | Numeric |
| `fat` | Fat content (grams) | Numeric |
| `carbs` | Carbohydrate content (grams) | Numeric |
| `protein` | Protein content (grams) | Numeric |
| `avg_rating` | Average user rating (0–5) | Numeric |
| `prep_time` | Preparation time (minutes) | Numeric |
| `cook_time` | Cooking time (minutes) | Numeric |
| `servings` | Number of servings | Numeric |

## Task #1: Load and Explore the Data

Load `cuisines_clean.csv` into a DataFrame called `cuisines_df`.

1. Print the **shape** of the DataFrame.
2. Display the **first 5 rows**.
3. Print the **top 10 countries** by number of recipes using `value_counts()`.
4. Print **summary statistics** for the numeric columns using `.describe()`.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #1 in this cell

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load the data
cuisines_df = pd.read_csv('cuisines_clean.csv')

# 1. Print shape
print(f'Shape: {cuisines_df.shape}')

# 2. Display first 5 rows
display(cuisines_df.head(...))

# 3. Top 10 countries by recipe count
print('\nTop 10 countries by recipe count:')
print(cuisines_df['country'].value_counts().head(...))

# 4. Summary statistics
print('\nSummary statistics:')
display(cuisines_df.describe())

## Task #2: Select Features and Standardize

Hierarchical clustering is sensitive to the scale of variables. Features measured in large units (e.g., calories in the hundreds) will dominate features measured in small units (e.g., average rating on a 0–5 scale) unless we standardize first.

1. Define `feature_cols` as the list below and create `X_cluster = cuisines_df[feature_cols].copy()`.

   ```python
   feature_cols = ['calories', 'fat', 'carbs', 'protein', 'avg_rating', 'prep_time', 'cook_time', 'servings']
   ```

2. Drop any rows with missing values using `.dropna()`. Store the cleaned subset as `X_cluster` and print its shape.

3. Standardize using `StandardScaler`: fit **only** on `X_cluster` and transform to produce `X_scaled` (a NumPy array).

4. Print the **mean** and **standard deviation** of each scaled column (they should be approximately 0 and 1).

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #2 in this cell

from sklearn.preprocessing import StandardScaler

# 1. Select features
feature_cols = [...]
X_cluster = cuisines_df[...].copy()

# 2. Drop missing values
X_cluster = X_cluster.dropna()
print(f'Shape after dropping missing values: {X_cluster.shape}')

# 3. Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(...)

# 4. Check means and standard deviations
print(f'\nMean of scaled features (should be ~0):')
print(pd.Series(X_scaled.mean(axis=0).round(4), index=feature_cols))
print(f'\nStd of scaled features (should be ~1):')
print(pd.Series(X_scaled.std(axis=0).round(4), index=feature_cols))

## Task #3: Compute the Linkage Matrix and Plot the Dendrogram

A **dendrogram** is a tree diagram that shows how individual observations are merged step by step into larger and larger clusters. The height at which two branches merge reflects the **distance** between them — taller merges mean more dissimilar groups.

1. Import `linkage` and `dendrogram` from `scipy.cluster.hierarchy`.
2. Compute the linkage matrix `Z` using **Ward's method** (`method='ward'`) on `X_scaled`.
3. Plot the dendrogram, **truncated to the last 30 merges** (use `truncate_mode='lastp'`, `p=30`). Add axis labels and a title.
4. Add a **horizontal dashed line** at the cut height you think best separates the clusters, using `plt.axhline()`.
5. Compute **silhouette scores** for k = 2 to 10 using `silhouette_score` from `sklearn.metrics` and `fcluster` from `scipy.cluster.hierarchy`. Print the scores, identify the best k, and plot the scores as a line chart.

In [ ]:
# Place your answer for Task #3 in this cell

from scipy.cluster.hierarchy import linkage, dendrogram

# Compute Ward linkage matrix
Z = linkage(X_scaled, method='ward')

k = 4

# Plot the truncated dendrogram
plt.figure(figsize=(12, 5), dpi=150)
dendrogram(
    Z,
    truncate_mode='lastp',
    p=30,
    show_leaf_counts=True,
    leaf_rotation=90,
    color_threshold=0.7 * max(Z[:, 2])
)
plt.xlabel('Cluster (number of recipes in parentheses)')
plt.ylabel('Ward Linkage Distance')
plt.title('Hierarchical Clustering Dendrogram')
plt.axhline(y=..., color='red', linestyle='--', linewidth=1.5, label='Chosen cut')
plt.legend()
plt.tight_layout()
plt.show()

# Silhouette scores for k = 2–10
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import fcluster

k_range = range(2, 11)
sil_scores = {}
for k in k_range:
    labels = fcluster(Z, t=k, criterion='maxclust')
    sil_scores[k] = ...

best_k = max(sil_scores, key=sil_scores.get)
print('Silhouette scores by k:')
for k, s in sil_scores.items():
    marker = '  <-- best' if k == best_k else ''
    print(f'  k={k}: {s:.4f}{marker}')

plt.figure(figsize=(6, 3), dpi=150)
plt.plot(list(sil_scores.keys()), list(sil_scores.values()), marker='o', color='coral')
plt.axvline(x=best_k, color='red', linestyle='--', linewidth=1.2, label=f'Best k={best_k}')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette score')
plt.title('Silhouette Scores for k = 2–10')
plt.xticks(list(k_range))
plt.legend()
plt.tight_layout()
plt.show()

## Task #4: Cut the Dendrogram and Assign Cluster Labels

Use `fcluster` to cut the dendrogram using the best k identified by the silhouette score in Task #3.

1. Import `fcluster` from `scipy.cluster.hierarchy`.
2. Use `best_k` from Task #3 as the number of clusters. Call `fcluster(Z, t=best_k, criterion='maxclust')` to obtain cluster labels (1-indexed). Store as `cluster_labels`.
3. Create `cuisines_clustered` by taking the rows of `cuisines_df` that correspond to `X_cluster` (use `.loc[X_cluster.index]`) and adding a `'cluster'` column.
4. Print the **size** of each cluster using `value_counts()`, sorted by cluster number.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #4 in this cell

from scipy.cluster.hierarchy import fcluster

# 1 & 2. Cut dendrogram using best_k from silhouette scores
cluster_labels = fcluster(Z, t=..., criterion=...)

# 3. Add cluster labels to a copy of the relevant rows
cuisines_clustered = cuisines_df.loc[X_cluster.index].copy()
cuisines_clustered['cluster'] = ...

# 4. Print cluster sizes
print('Cluster sizes:')
print(cuisines_clustered['cluster'].value_counts().sort_index())

## Task #5: Profile the Clusters

To interpret the clusters we identified, we compute the **mean value of each feature** within each cluster and visualize the result as a heatmap.

1. Compute the mean of each column in `feature_cols` grouped by `'cluster'`. Store as `cluster_means` and display it.
2. Rescale `cluster_means` back to standardized units using `scaler.transform()`. Store as a DataFrame called `cluster_means_scaled` with the same index and column names.
3. Create a **heatmap** of `cluster_means_scaled` using `sns.heatmap()` with `annot=True`, `cmap='RdBu_r'`, and `center=0`. Label rows as `'Cluster 1'`, `'Cluster 2'`, etc.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #5 in this cell

import seaborn as sns

# 1. Compute mean features by cluster
cluster_means = cuisines_clustered.groupby('cluster')[...].mean().round(2)
print('Mean feature values by cluster:')
display(cluster_means)

# 2. Rescale cluster means to standardized units
cluster_means_scaled = pd.DataFrame(
    scaler.transform(...),
    index=cluster_means.index,
    columns=feature_cols
)

# 3. Heatmap
plt.figure(figsize=(10, 4), dpi=150)
sns.heatmap(
    cluster_means_scaled,
    annot=...,
    fmt='.2f',
    cmap=...,
    center=...,
    xticklabels=feature_cols,
    yticklabels=[f'Cluster {i}' for i in cluster_means.index]
)
plt.title('Standardized Cluster Means (Red = High, Blue = Low)')
plt.tight_layout()
plt.show()

## Task #6: Visualize Clusters with Scatter Plots

Create **two side-by-side scatter plots** to visualise the clusters in feature space.

- **Left plot:** `calories` (x-axis) vs `protein` (y-axis), with points coloured by cluster.
- **Right plot:** `fat` (x-axis) vs `carbs` (y-axis), with points coloured by cluster.

Use `alpha=0.5` and `s=20` for the scatter points. Add axis labels, titles, and a legend to each plot.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #6 in this cell

cmap = plt.get_cmap('tab10')
colors = {k: cmap(i) for i, k in enumerate(sorted(cuisines_clustered['cluster'].unique()))}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

for c in sorted(cuisines_clustered['cluster'].unique()):
    mask = cuisines_clustered['cluster'] == c

    # Left plot: calories vs protein
    axes[0].scatter(
        cuisines_clustered.loc[mask, ...],
        cuisines_clustered.loc[mask, ...],
        alpha=0.5, s=20, label=f'Cluster {c}', color=colors[c]
    )

    # Right plot: fat vs carbs
    axes[1].scatter(
        cuisines_clustered.loc[mask, ...],
        cuisines_clustered.loc[mask, ...],
        alpha=0.5, s=20, label=f'Cluster {c}', color=colors[c]
    )

axes[0].set_xlabel('Calories')
axes[0].set_ylabel('Protein (g)')
axes[0].set_title('Calories vs Protein by Cluster')
axes[0].legend()

axes[1].set_xlabel('Fat (g)')
axes[1].set_ylabel('Carbs (g)')
axes[1].set_title('Fat vs Carbs by Cluster')
axes[1].legend()

plt.tight_layout()
plt.show()

## Task #7: Cuisine Breakdown by Cluster

Explore which cuisines are most represented in each cluster.

1. Use `groupby` on `['cluster', 'country']` and `.size()` to count recipes per country within each cluster. Reset the index and name the count column `'count'`.
2. Sort so that within each cluster, countries are ordered from most to fewest recipes.
3. Use `.groupby('cluster').head(5)` to keep the **top 5 countries per cluster** and reset the index. Store as `top_countries`.
4. Display `top_countries`.

Fill in the `...` in the code below.

In [ ]:
# Place your answer for Task #7 in this cell

# Count recipes per country within each cluster
country_by_cluster = (
    cuisines_clustered
    .groupby([..., ...])
    .size()
    .reset_index(name='count')
    .sort_values(['cluster', 'count'], ascending=[True, False])
)

# Top 5 countries per cluster
top_countries = (
    country_by_cluster
    .groupby('cluster')
    .head(...)
    .reset_index(drop=True)
)

display(top_countries)

---

## Question #1: Dendrogram Interpretation and Cluster Choice (5 marks)

Answer the following based on your dendrogram and silhouette plot from Task #3 and your cluster assignments from Task #4. Your answer should address **all five** of the following for full marks.

1. **Describe the dendrogram.** Where does the largest "jump" in linkage distance appear? What does this suggest about the natural number of clusters in the data? (1 mark)

2. **Explain Ward's method.** What quantity does Ward's linkage minimise at each merge step? How might the dendrogram look different if we had used **single linkage** instead? (1 mark)

3. **Justify standardization.** Why is it important to standardize the features before clustering? What could go wrong if we clustered on the raw (unstandardized) features from this dataset? Give a concrete example using two of the features. (1 mark)

4. **Sensitivity to cut height.** If you cut the dendrogram at a Ward Linkage Distance of just under 40, how many clusters would you obtain? How would this have affected the cluster profiles and your interpretation compared to the cut you chose? (1 mark)

5. **Silhouette score vs. dendrogram.** Does the k with the highest silhouette score agree with the k you chose from the dendrogram? If they differ, which would you trust more for this dataset and why? (1 mark)

---

Place your answer for Question #1 in this cell

---

---

## Question #2: Cluster Interpretation (4 marks)

Answer the following based on your cluster profile heatmap from Task #5, the scatter plots from Task #6, and the country breakdown from Task #7. Your answer should address **all four** of the following for full marks.

1. **Name each cluster.** Give each of the clusters a brief, descriptive name based on its nutritional and timing profile from the heatmap. Explain your reasoning for at least two of the names. (1 mark)

2. **Connect clusters to dishes.** Choose one cluster and name two specific dishes from a real cuisine that you would expect to fall into it. Explain why, using the cluster's feature values. (1 mark)

3. **Evaluate the cuisine breakdown.** Does the cuisine breakdown in Task #7 match what you would expect given each cluster's profile? Give one specific example where the match is **surprising or noteworthy** and explain why. (1 mark)

4. **Identify a limitation.** Identify one limitation of using hierarchical clustering for this recipe dataset and explain how it might affect the conclusions you can draw. (1 mark)

---

Place your answer for Question #2 in this cell

---